# Schema Evolution Demo — Adding `discount_code`

This notebook demonstrates and documents the schema-evolution scenario for the checkpoint demo: `discount_code` starts appearing on real order events partway through the stream, and the pipeline keeps running — no restart, no redeploy, no code change on the consumer side.

## How it's handled
`stream_orders_to_bronze` declares `discount_code` as a **nullable field in the Spark schema from the start**, before any event actually contains it. That's a deliberate choice over dynamic per-microbatch schema inference:

- The stream never needs to stop or restart when the field starts arriving — it was already expecting it.
- `from_json` simply returns `null` for a declared field that's missing from a given JSON payload, and returns the real value once it's present — no special handling needed.
- This mirrors a realistic pattern: agree on the target schema with the producer side ahead of time, mark new fields nullable, and both sides ship independently without coordinating a deploy.

Honest tradeoff: this only works because the field name was known in advance. A truly unplanned new column (one nobody declared anywhere) would need Auto Loader-style rescued-data handling or a `foreachBatch` + Delta `mergeSchema` approach instead — the technique we used in Lab 3 for file-based ingestion. For a single, planned business field like a discount code, pre-declaring it is simpler and just as valid, and is a common real-world tradeoff between planned and unplanned schema drift.

In [0]:
%sql
DESCRIBE TABLE dbr_dev.brazilian_ecommerce_bronze.brz_orders

`discount_code` is already a column in the table, it was there from the very first write, before any event ever populated it. That's the core of what's being demonstrated: the schema didn't change when the data changed, because it was already wide enough to hold it.

## Step 1 — Before: orders without discount codes
In `01_order_event_producer`, set the `include_discount_code` widget to `false` and run it while `stream_orders_to_bronze` is actively running. Then check the table:

In [0]:
%sql
SELECT
    order_id,
    product_id,
    price,
    discount_code,
    ingestion_timestamp
FROM dbr_dev.brazilian_ecommerce_bronze.brz_orders
ORDER BY ingestion_timestamp DESC

Expected: `discount_code` is `NULL` on every row — the stream is running fine, it just hasn't seen the field populated yet.

## Step 2 — The evolution moment (live in the demo)
Go back to `01_order_event_producer`, flip `include_discount_code` to `true`, and rerun it. **`stream_orders_to_bronze` stays running the whole time — do not stop or restart it.** This is the moment to call out explicitly during the presentation.

## Step 3 — After: new rows carry the new column, old rows don't

In [0]:
%sql
SELECT
    order_id,
    product_id,
    price,
    discount_code,
    ingestion_timestamp
FROM dbr_dev.brazilian_ecommerce_bronze.brz_orders
WHERE discount_code IS NOT NULL
ORDER BY ingestion_timestamp DESC

## Step 4 — Quantify the transition
A clean visual for the demo: how many rows fall on each side of the change.

In [0]:
%sql
SELECT
    CASE
        WHEN discount_code IS NULL THEN 'Before schema change'
        ELSE 'After schema change'
    END AS event_type,
    COUNT(*) AS orders
FROM dbr_dev.brazilian_ecommerce_bronze.brz_orders
GROUP BY
    CASE
        WHEN discount_code IS NULL THEN 'Before schema change'
        ELSE 'After schema change'
    END;

## Step 5 — Confirm the stream itself never stopped
Check `query.status` / `query.lastProgress` in `stream_orders_to_bronze` (or the Streaming tab in the Spark UI) across the whole demo window. `batchId` should keep incrementing continuously through Steps 1–3 with no gap or error — that's the actual proof the schema change never touched the running query, not just that the data looks right afterward.
